In [ ]:
from astropy.io import fits
from astropy.table import Table
from matplotlib import pyplot as plt
import numpy as np

In [ ]:
master_cat = Table.read('COSMOSWeb_mastercatalog_v1_photom_primary.fits')[:2]
master_cat

In [ ]:
master_cat_morph = Table.read('COSMOSWeb_mastercatalog_v1_ml_morph.fits')[:4]
master_cat_morph#['morph_flag_f150w']

In [ ]:
for column_names in master_cat.columns:
    print(column_names)

In [ ]:
tiles = np.unique(master_cat['tile'])

def get_image_cutout(image, x_center, y_center, size=50):
    x_center = int(round(x_center))
    y_center = int(round(y_center))
    half_size = size // 2
    return image[y_center - half_size:y_center + half_size,
                      x_center - half_size:x_center + half_size]

for tile in tiles[10:11]:
    mask = (master_cat['tile'] == tile) & (master_cat['flag_blend']==False)
    tile_cat = master_cat[master_cat['tile'] == tile]
    obj_id = np.random.choice(tile_cat['id'])
    obj = tile_cat[tile_cat['id'] == obj_id][0]
    with fits.open(f'detection_images/detection_chi2pos_SWLW_{tile}.fits') as image_lis:
        with fits.open(f'segmentation_maps/detection_chi2pos_SWLW_{tile}_segmap_v1.3.fits.gz') as seg_lis:
            image = image_lis[0]
            seg   = seg_lis[0]

            x_center = obj['x_image']
            y_center = obj['y_image']
            image_cut = get_image_cutout(image.data, x_center, y_center, size=164)
            seg_cut = get_image_cutout(np.where(seg.data==obj['segment-id'],1,0), x_center, y_center, size=64)
            plt.imshow(image_cut, origin='lower', cmap='gray', vmin=0, vmax=np.percentile(image_cut, 99))
            plt.title(f"Tile: {tile}, ID: {obj['id']}")
            plt.colorbar(label='Flux')
            plt.show()
            plt.imshow(seg_cut, origin='lower', cmap='gray',vmin=0,vmax=1)
            plt.title(f"seg: Tile: {tile}, ID: {obj['id']}")
            plt.show()
            print()
    #now check with f115w and f150w
    for filter in ['f115w', 'f150w']:
        with fits.open(f'{filter}/mosaic_nircam_{filter}_COSMOS-Web_30mas_{tile}_v1.0_sci.fits') as sci_lis:
            image = sci_lis[0]
            image_cut = get_image_cutout(image.data, x_center, y_center, size=64)
            plt.imshow(image_cut, origin='lower', cmap='gray', vmin=0, vmax=np.percentile(image_cut, 99))
            plt.title(f"{filter}: Tile: {tile}, ID: {obj['id']}")
            plt.colorbar(label='Flux')
            plt.show()
            print(obj[f'snr_{filter}'])
            
            

In [ ]:
with fits.open('f150w/mosaic_nircam_f150w_COSMOS-Web_30mas_A10_v1.0_sci.fits') as hdul:
    print(hdul[0].data.shape)
with fits.open('detection_images/detection_chi2pos_SWLW_A10.fits') as hdul:
    print(hdul[0].data.shape)

In [ ]:
plt.hist(master_cat['snr_f115w'],cumulative=True, range = (0,10),bins=50)

In [ ]:
mask = (master_cat['flag_blend'] != True) & (master_cat['warn_flag']<=2)

len(master_cat[mask])

In [ ]:
import h5py
data = h5py.File('../jwst/f115w/f115w_A1.h5')
data.keys()

In [ ]:
print(data.keys())
index = 8550
print(data['snr_f115w'][index])
plt.imshow(data['image'][index])
print(data['image'][index].shape)

In [ ]:
plt.scatter(np.array(-2.5*np.log10(data['flux_auto_f115w'])),np.array(data['snr_f115w']),alpha=0.3,s=0.4)
plt.plot((-5,15),(3,3),c='red')
plt.yscale('log')

In [ ]:
images = np.array(data['image'])

In [ ]:
snr = np.array(data['snr_f115w'])
len(snr[snr>2])*20

In [ ]:
flux = np.array(data['flux_auto_f115w'])
len(flux[flux<=0])*20

In [ ]:
master_cat['kron1_a'].max()

In [ ]:
plt.hist(master_cat['kron1_b'])
plt.yscale('log')

In [ ]:
from astroclip.astrodino.data.loaders import make_dataset
from astroclip.astrodino.data.augmentations import ToRGB
from torchvision import transforms


In [ ]:
crop_size=64
transform = transforms.Compose(
    [
        transforms.CenterCrop(crop_size),
        ToRGB(),
    ]
)
dataset_str = "jwst:split=train:root=/ptmp/yacheng/outthere_ssl/images/jwst:filter=f115w"
jwst = make_dataset(
dataset_str = dataset_str,
transform = transform,
channel = 2
)


In [ ]:
from matplotlib import pyplot as plt
data = jwst[5]
plt.imshow(data[0][0])
print(data)

In [ ]:
plt.hist(jwst[2][0][0],bins=60)
plt.show()

In [ ]:
jwst[0

In [ ]:
import os
os.chdir('/u/yacheng/projects/ssl_outthere/data/survey/cosmos_2025')

# Read both catalogs
cat_primary = Table.read('COSMOSWeb_mastercatalog_v1_photom_primary.fits')
cat_morph = Table.read('COSMOSWeb_mastercatalog_v1_ml_morph.fits')

print(f"Primary catalog length: {len(cat_primary)}")
print(f"Morph catalog length: {len(cat_morph)}")
print(f"Lengths match: {len(cat_primary) == len(cat_morph)}")

# Combine by columns (hstack)
cat_combined = cat_primary.copy()
# Add all morph columns except those that might conflict
for col_name in cat_morph.colnames:
    if col_name not in cat_combined.colnames:
        cat_combined[col_name] = cat_morph[col_name]

print(f"\nCombined catalog shape: {len(cat_combined)} rows x {len(cat_combined.colnames)} columns")
print(f"Total columns: {len(cat_combined.colnames)}")
print(f"\nFirst few column names: {cat_combined.colnames[:10]}")


In [ ]:
# Get all unique tiles from primary catalog
available_fields = np.unique(cat_primary['tile'])
print(f"Available tiles/fields: {available_fields}")
print(f"Number of unique tiles: {len(available_fields)}")
